In [ ]:
%cd /content/drive/MyDrive/실전프로젝트/데이터 분석

In [ ]:
import json
import random
from pathlib import Path
from collections import defaultdict
from math import floor

# =========================
# 설정
# =========================

INPUT_FILE = "./questions_raw_draft_image_path_null.json"

OUTPUT_SPLIT_FILE = "questions_split_3_7.json"
OUTPUT_TEST_FILE = "questions_test.json"
OUTPUT_TRAIN_FILE = "questions_train.json"

TEST_RATIO = 0.3
RANDOM_SEED = 42

random.seed(RANDOM_SEED)


# =========================
# 파일 읽기
# =========================

input_path = Path(INPUT_FILE)

with input_path.open("r", encoding="utf-8") as f:
    questions = json.load(f)

print(f"전체 문항 수: {len(questions)}")


# =========================
# 유형별로 묶기
# =========================

type_groups = defaultdict(list)

for q in questions:
    type_groups[q["question_type_id"]].append(q)


# =========================
# 하위분류별 비율 유지해서 test 개수 배정
# =========================

def allocate_test_counts_by_subcategory(items, test_total):
    """
    한 문항 유형 안에서 sub_category별 문항 수 비율을 최대한 유지하면서
    검사용 문항 수를 배정하는 함수.
    """

    sub_groups = defaultdict(list)

    for q in items:
        sub_category = q.get("sub_category", "기타")
        sub_groups[sub_category].append(q)

    total_count = len(items)

    allocations = {}
    remainders = []

    # 1차 배정: 비율대로 계산 후 내림
    for sub_category, sub_items in sub_groups.items():
        raw_count = len(sub_items) / total_count * test_total
        base_count = floor(raw_count)

        allocations[sub_category] = base_count
        remainders.append((raw_count - base_count, sub_category))

    # 남은 개수 보정
    current_total = sum(allocations.values())
    remaining = test_total - current_total

    # 소수점 나머지가 큰 하위분류부터 1개씩 추가
    remainders.sort(reverse=True)

    for _, sub_category in remainders:
        if remaining <= 0:
            break

        # 해당 하위분류 문항 수보다 많이 배정하지 않도록 제한
        if allocations[sub_category] < len(sub_groups[sub_category]):
            allocations[sub_category] += 1
            remaining -= 1

    return sub_groups, allocations


# =========================
# test/train 분할
# =========================

split_questions = []
test_questions = []
train_questions = []

for question_type_id, items in sorted(type_groups.items()):
    items = items[:]

    total_count = len(items)
    test_total = round(total_count * TEST_RATIO)

    sub_groups, allocations = allocate_test_counts_by_subcategory(items, test_total)

    print()
    print(f"[유형 {question_type_id}] 전체 {total_count}개 / 검사용 {test_total}개 / 훈련용 {total_count - test_total}개")

    for sub_category, sub_items in sorted(sub_groups.items()):
        sub_items = sub_items[:]
        random.shuffle(sub_items)

        sub_test_count = allocations[sub_category]

        print(f"  - {sub_category}: 전체 {len(sub_items)}개 / 검사용 {sub_test_count}개")

        for idx, q in enumerate(sub_items):
            new_q = q.copy()

            if idx < sub_test_count:
                new_q["question_purpose"] = "test"
                test_questions.append(new_q)
            else:
                new_q["question_purpose"] = "train"
                train_questions.append(new_q)

            split_questions.append(new_q)


# =========================
# 정렬
# 보기 좋게 유형 → 원본 번호 순으로 정렬
# =========================

def sort_key(q):
    return (
        q.get("question_type_id", 999),
        q.get("source_number", 999)
    )

split_questions.sort(key=sort_key)
test_questions.sort(key=sort_key)
train_questions.sort(key=sort_key)


# =========================
# 결과 검증
# =========================

print()
print("===== 최종 분할 결과 =====")
print(f"전체 문항 수: {len(split_questions)}")
print(f"검사용 문항 수: {len(test_questions)}")
print(f"훈련용 문항 수: {len(train_questions)}")

print()
print("===== 유형별 개수 =====")

final_count_by_type = defaultdict(lambda: {"test": 0, "train": 0})

for q in split_questions:
    q_type = q["question_type_name"]
    purpose = q["question_purpose"]
    final_count_by_type[q_type][purpose] += 1

for q_type, counts in final_count_by_type.items():
    print(f"{q_type}: 검사용 {counts['test']}개 / 훈련용 {counts['train']}개")


# =========================
# 파일 저장
# =========================

with open(OUTPUT_SPLIT_FILE, "w", encoding="utf-8") as f:
    json.dump(split_questions, f, ensure_ascii=False, indent=2)

with open(OUTPUT_TEST_FILE, "w", encoding="utf-8") as f:
    json.dump(test_questions, f, ensure_ascii=False, indent=2)

with open(OUTPUT_TRAIN_FILE, "w", encoding="utf-8") as f:
    json.dump(train_questions, f, ensure_ascii=False, indent=2)

print()
print("저장 완료")
print(f"- 전체 분할 파일: {OUTPUT_SPLIT_FILE}")
print(f"- 검사용 파일: {OUTPUT_TEST_FILE}")
print(f"- 훈련용 파일: {OUTPUT_TRAIN_FILE}")